# Module 5 · Structured Output & Streaming
Force JSON responses with schemas. Stream tokens in real-time.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'models/gemma-4-26b-a4b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Setup complete. Model: models/gemma-4-26b-a4b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [4]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 8. Structured Output (JSON)

Force the model to output valid JSON matching a schema — essential for production pipelines.

In [6]:
invoice_text = """
Invoice from Acme Corp dated March 15 2024.
Customer: TechStartup Inc, 500 Main St, San Francisco CA.
Items: 10 units of Widget Pro at $49.99 each, plus shipping $12.50.
Total due: $512.40. Payment due within 30 days.
"""

schema = {
    "type": "object",
    "properties": {
        "vendor": {"type": "string"},
        "customer": {"type": "string"},
        "date": {"type": "string"},
        "items": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "description": {"type": "string"},
                    "quantity": {"type": "number"},
                    "unit_price": {"type": "number"}
                }
            }
        },
        "shipping": {"type": "number"},
        "total": {"type": "number"},
        "payment_due_days": {"type": "number"}
    }
}

r = client.models.generate_content(
    model=MODEL,
    contents=f"Extract invoice details as JSON:\n{invoice_text}",
    config=cfg(temperature=0.0, response_mime_type="application/json", response_schema=schema)
)

data = json.loads(get_text(r))
print(json.dumps(data, indent=2))

{
  "vendor": "Acme Corp",
  "customer": "TechStartup Inc, 500 Main St, San Francisco CA",
  "date": "March 15 2024",
  "items": [
    {
      "description": "Widget Pro",
      "quantity": 10,
      "unit_price": 49.99
    }
  ],
  "shipping": 12.5,
  "total": 512.4,
  "payment_due_days": 30
}


In [7]:
# The result is real Python data — access fields directly
print(f"Vendor  : {data['vendor']}")
print(f"Customer: {data['customer']}")
print(f"Total   : ${data['total']}")
for item in data.get('items', []):
    print(f"  - {item.get('description','?')}: {item.get('quantity','?')} × ${item.get('unit_price','?')}")

Vendor  : Acme Corp
Customer: TechStartup Inc, 500 Main St, San Francisco CA
Total   : $512.4
  - Widget Pro: 10 × $49.99


---
## 9. Streaming Responses

By default the API waits until the **full response is ready** before returning. Streaming sends tokens back as they're generated — like ChatGPT's typing effect.

```
Non-streaming:  [........generating........] → receive everything at once
Streaming:      [token] [token] [token] ...  → receive as produced
```

Use streaming whenever the response will be longer than a sentence, or users are waiting for output.

In [8]:
# Non-streaming — we wait until everything is done
start = time.time()
r = client.models.generate_content(
    model=MODEL,
    contents="Write a 4-line poem about Python programming.",
    config=cfg(temperature=0.8)
)
elapsed = time.time() - start
print(f"Non-streaming: received after {elapsed:.1f}s")
print(get_text(r))

Non-streaming: received after 25.4s
With whitespace and syntax, so simple and clean,
The clearest of logic that ever was seen.
With modules to import and libraries vast,
A language of power, built truly to last.


In [7]:
# Streaming — tokens arrive in real-time, print as they come
print("Streaming (tokens arrive in real-time):")
print("-" * 40)

chunk_count = 0
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="Write a 4-line poem about Python programming.",
    config=cfg(temperature=0.8)
):
    # Gemma 4 streams thinking chunks (thought=True) before answer chunks
    # We skip thought chunks so only the final answer prints
    if chunk.candidates:
        for part in chunk.candidates[0].content.parts:
            if not getattr(part, 'thought', False) and part.text:
                print(part.text, end="", flush=True)
                chunk_count += 1

print(f"\n\nReceived {chunk_count} answer chunks")

Streaming (tokens arrive in real-time):
----------------------------------------


Clean indentation, a readable grace

,
Solving a problem at a rapid pace.
With libraries vast and a syntax so lean,
The sleekest

 language the world's ever seen.



Received 3 answer chunks


In [8]:
# Collecting streamed output into a single string (common pattern in apps)
full_text = ""
for chunk in client.models.generate_content_stream(
    model=MODEL,
    contents="List 3 Python best practices, numbered.",
    config=cfg(temperature=0.3)
):
    if chunk.candidates:
        for part in chunk.candidates[0].content.parts:
            if not getattr(part, 'thought', False) and part.text:
                full_text += part.text
                print(part.text, end="", flush=True)

print(f"\n\nTotal characters collected: {len(full_text)}")

1. **Follow PEP 8:** Adhere to the official Python style guide to ensure your code is readable and consistent

. This includes using 4 spaces per indentation level and using `snake_case` for functions and variables.
2

. **Use Virtual Environments:** Always use tools like `venv` or `conda` to isolate project dependencies. This prevents

 version conflicts between different projects and keeps your global Python installation clean.
3. **Implement Type Hinting:** Use type

 hints (e.g., `def add(a: int, b: int) -> int:`) to make

 your code more self-documenting. This helps other developers understand your intent and allows IDEs to catch potential bugs before

 the code is even run.



Total characters collected: 676
